#FlowerVision – Sistema di riconoscimento automatico dei fiori per AgriTech

## Contesto iniziale
GreenTech Solutions Ltd. opera nel settore **AgriTech** e vuole automatizzare il riconoscimento dei fiori per migliorare il monitoraggio delle colture, il controllo della salute vegetale e la qualità delle decisioni agronomiche.

Il riconoscimento automatico di immagini floreali permette di:
- ridurre il tempo dedicato all'identificazione manuale;
- aumentare la coerenza delle classificazioni;
- supportare processi decisionali basati sui dati;
- migliorare efficienza operativa e sostenibilità.

Il dataset del progetto contiene due classi:
- **Daisy** (Margherita)
- **Dandelion** (Tarassaco)

## Obiettivo del progetto
L'obiettivo principale è sviluppare un modello di **Deep Learning** capace di classificare automaticamente i fiori con il miglior **F1-score macro** possibile sul dataset di test.

Per raggiungere questo obiettivo il notebook utilizza:
- **PyTorch** per il training;
- **timm** per il **transfer learning** e le **data augmentation**;
- un approccio sperimentale con più configurazioni;
- una pipeline ordinata, commentata e facilmente estendibile.

## Metodologia adottata

Per affrontare il problema di classificazione automatica dei fiori, è stato adottato un approccio basato su **Deep Learning** e **transfer learning** con la libreria **timm** di PyTorch.

L’idea principale è sfruttare un modello pre-addestrato su un dataset ampio e generico, per poi adattarlo al problema specifico di classificazione tra le due classi presenti nel dataset:

- **Daisy (Margherita)**
- **Dandelion (Tarassaco)**

Questo approccio è stato scelto perché consente di:

- ridurre i tempi di addestramento
- ottenere buone prestazioni anche con dataset di dimensioni contenute
- sfruttare rappresentazioni visive già apprese dal modello durante il pretraining

La pipeline sperimentale è stata organizzata nelle seguenti fasi:

1. **Preparazione del dataset** con suddivisione in train, validation e test
2. **Definizione delle trasformazioni** e delle tecniche di data augmentation
3. **Costruzione del modello** con backbone pre-addestrato
4. **Addestramento e validazione** del modello
5. **Selezione della configurazione migliore** sulla base del validation set
6. **Valutazione finale sul test set** mediante metriche di classificazione

La metrica principale scelta per il confronto tra gli esperimenti è il **F1-score macro**, in quanto più adatto della sola accuracy a fornire una valutazione equilibrata delle prestazioni del modello sulle due classi.

## Struttura del notebook

Il notebook è organizzato in 8 blocchi principali:

1. Setup ambiente  
2. Import delle librerie  
3. Seed e device  
4. Configurazione centralizzata  
5. Classe `ExperimentRunner`  
6. Classe `Evaluator`  
7. Definizione degli esperimenti  
8. Confronto finale e valutazione sul test set  

## Tecniche testate

### Esperimento 1 — Baseline pulita
- modello pretrained
- backbone congelato
- augmentation base
- testa finale allenata

### Esperimento 2 — Fine-tuning progressivo
- prime epoche con backbone congelato
- sblocco graduale degli ultimi layer
- augmentation moderata

### Esperimento 3 — Fine-tuning completo
- tutto il backbone allenabile
- augmentation moderata
- label smoothing

### Esperimento 4 — Augmentation più forte
- fine-tuning completo
- augmentation più intensa
- regolarizzazione più robusta

> Nota metodologica: il **validation set** viene usato per confrontare gli esperimenti; il **test set** va usato solo alla fine sul modello migliore scelto tramite validation.


In [ ]:

# ============================================================
# CELLA 1 — Download e preparazione del dataset
# ============================================================
# Il dataset è ospitato su S3 e viene scaricato solo se non è già presente
# in locale, evitando download ripetuti ad ogni esecuzione.
# Su Google Colab il file viene salvato nella RAM temporanea della sessione.
# ============================================================

import os

DATASET_URL = "https://proai-datasets.s3.eu-west-3.amazonaws.com/progetto-finale-flowes.tar.gz"
DATASET_ARCHIVE = "progetto-finale-flowes.tar.gz"
DATASET_FOLDER = "progetto-finale-flowes"

if not os.path.isdir(DATASET_FOLDER):
    print("Dataset non trovato in locale. Avvio download...")
    # !wget scarica il file dall'URL e lo salva come DATASET_ARCHIVE
    !wget -O {DATASET_ARCHIVE} {DATASET_URL}
    # !tar estrae l'archivio .tar.gz nella directory corrente
    !tar -xzf {DATASET_ARCHIVE}
    print("Download e estrazione completati.")
else:
    print("Dataset già presente:", DATASET_FOLDER)


In [ ]:

# Spostiamo la cartella del dataset in una sottodirectory "datasets/" per
# mantenere la struttura del progetto più ordinata.
# -p evita errori se la directory esiste già.
! mkdir -p datasets
! mv {DATASET_FOLDER} datasets/{DATASET_FOLDER}


In [ ]:

# ============================================================
# CELLA 2 — Import delle librerie
# ============================================================

import copy          # copy.deepcopy() per clonare configurazioni senza modificare l'originale
import os            # operazioni su file e directory
import random        # per impostare il seed Python standard
import time          # misura il tempo di ogni epoca

from dataclasses import dataclass  # sintassi compatta per classi di configurazione

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch: il framework principale per la costruzione e il training della CNN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# ImageFolder: carica automaticamente le immagini organizzate in sotto-cartelle per classe
# es. train/daisy/, train/dandelion/ → classe inferita dal nome della cartella
from torchvision.datasets import ImageFolder

# timm (PyTorch Image Models): raccolta di modelli pretrained e utility per transfer learning
import timm
from timm.data import create_transform, resolve_data_config
from timm.data.mixup import Mixup                          # tecnica di data augmentation Mixup
from timm.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy  # loss per regolarizzazione

# sklearn: metriche di valutazione standard
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)


In [ ]:

# ============================================================
# CELLA 3 — Seed e selezione del device
# ============================================================

def set_seed(seed: int = 42) -> None:
    """
    Imposta i seed su tutti i generatori di numeri casuali per garantire
    la riproducibilità degli esperimenti.

    PyTorch usa più sorgenti di casualità indipendenti:
    - random: operazioni Python standard
    - numpy: operazioni di array e sklearn (es. train_test_split)
    - torch (CPU): inizializzazione pesi e dropout
    - torch (GPU): operazioni su CUDA
    - cudnn.deterministic: forza CUDA a usare algoritmi deterministici
    - cudnn.benchmark=False: disabilita la selezione automatica algoritmo (non deterministica)

    Trade-off: deterministic=True rallenta leggermente il training ma garantisce
    che rieseguire il notebook due volte produca esattamente gli stessi risultati.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

# Selezione automatica del device:
# - su Google Colab con GPU abilitata → "cuda" (calcolo parallelo su migliaia di core)
# - in locale senza GPU               → "cpu"
# Le CNN traggono enorme vantaggio dalla GPU: speedup tipico 10-100x rispetto alla CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device in uso:", device)

if device.type == "cuda":
    print("GPU rilevata:", torch.cuda.get_device_name(0))


In [ ]:

# ============================================================
# CELLA 4 — Classe Config: configurazione centralizzata
# ============================================================
# Raccogliere tutti i parametri in un'unica struttura ha diversi vantaggi:
# 1. Evita di cercare i valori sparsi nel codice
# 2. Permette di fare copy.deepcopy(cfg) per variare un solo parametro
# 3. Rende esplicita la differenza tra esperimenti
# @dataclass genera automaticamente __init__, __repr__ ecc. dai campi annotati.
# ============================================================

@dataclass
class Config:
    """
    Classe di configurazione centrale del progetto.
    L'idea è raccogliere qui tutti i parametri modificabili dei vari test.
    """
    # --- Identificazione dell'esperimento ---
    experiment_name: str = "baseline"

    # --- Percorsi del dataset ---
    # La struttura attesa è: dataset_root/train/, dataset_root/valid/, dataset_root/test/
    # con sotto-cartelle per classe (es. daisy/, dandelion/)
    dataset_root: str = "progetto-finale-flowes"
    train_dir: str = "train"
    val_dir: str = "valid"
    test_dir: str = "test"

    # --- Modello pretrained ---
    # efficientnet_b0: ottimo compromesso tra accuratezza e velocità per dataset piccoli
    # Alternativa: resnet18 (più semplice), vit_small_patch16_224 (transformer-based)
    model_name: str = "efficientnet_b0"
    num_classes: int = 2           # daisy (0) e dandelion (1)

    # --- Iperparametri di training ---
    batch_size: int = 32           # 32 è un buon compromesso VRAM/stabilità del gradiente
    epochs: int = 15               # con early stopping raramente si arriva a 15
    lr: float = 1e-3               # lr iniziale: più alto per backbone congelato
    weight_decay: float = 1e-4     # regolarizzazione L2 su tutti i parametri
    num_workers: int = 2           # thread paralleli per il caricamento dati
    use_amp: bool = True           # Automatic Mixed Precision: dimezza l'uso di VRAM su GPU
    seed: int = 42

    # --- Strategia di fine-tuning ---
    freeze_backbone: bool = True         # se True: allena solo la testa finale
    progressive_unfreeze: bool = False   # se True: sblocca il backbone a unfreeze_epoch
    unfreeze_epoch: int = 4              # epoca in cui avviene lo sblocco progressivo

    # --- Scheduler del learning rate ---
    # False → ReduceLROnPlateau (riduce lr se il F1 non migliora per 2 epoche)
    # True  → CosineAnnealingLR (decadimento continuo a forma di coseno)
    use_cosine_scheduler: bool = False

    # --- Regolarizzazione avanzata ---
    # label_smoothing > 0: distribuisce la certezza del target su tutte le classi
    #   es. smoothing=0.1 → target 0.95 invece di 1.0 → previene l'overconfidence
    label_smoothing: float = 0.0
    # mixup_alpha > 0: crea immagini ibride (blend di due campioni) durante il training
    #   es. img_mix = 0.7*img1 + 0.3*img2, label_mix = 0.7*label1 + 0.3*label2
    mixup_alpha: float = 0.0

    # --- Livello di data augmentation ---
    # "basic"    → solo flip orizzontale (minima distorsione)
    # "moderate" → flip + color jitter + RandAugment leggero + random erasing
    # "strong"   → flip + vflip + color jitter forte + RandAugment intenso + random erasing
    augmentation: str = "basic"

    # --- Early stopping ---
    patience: int = 4              # epoche senza miglioramento prima dello stop

    # --- Salvataggio ---
    best_model_path: str = "best_flower_model.pth"


In [ ]:

# ============================================================
# CELLA 5 — ExperimentRunner: gestisce l'intero ciclo di vita di un esperimento
# ============================================================
# Questa classe centralizza tutte le operazioni necessarie per eseguire
# un esperimento: caricamento dati, costruzione modello, training, validazione.
# Separare la logica in metodi distinti rende più facile isolare e modificare
# singole componenti senza toccare il resto del pipeline.
# ============================================================

class ExperimentRunner:
    def __init__(self, config: Config, device=None):
        self.config = config
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # GradScaler è necessario per usare Automatic Mixed Precision (AMP) in modo stabile.
        # AMP usa float16 dove possibile per dimezzare la memoria GPU e accelerare il calcolo,
        # ma può portare a underflow numerici → lo scaler bilancia automaticamente la loss.
        self.scaler = torch.cuda.amp.GradScaler(
            enabled=(self.config.use_amp and self.device.type == "cuda")
        )

        # Loader e metadati del dataset: inizializzati in build_dataloaders()
        self.train_loader = None
        self.val_loader = None
        self.test_loader = None
        self.class_names = None

        # Componenti del modello: inizializzate nei rispettivi metodi build_*()
        self.model = None
        self.train_criterion = None       # loss usata durante il training (può includere smoothing/mixup)
        self.eval_criterion = nn.CrossEntropyLoss()  # loss usata in validation (sempre standard)
        self.optimizer = None
        self.scheduler = None
        self.mixup_fn = None

        # Configurazione delle trasformazioni del modello pretrained
        self.data_cfg = None

    # ----------------------------------------------------------
    # Utility
    # ----------------------------------------------------------

    def set_seed(self):
        set_seed(self.config.seed)

    def _check_dir(self, path: str) -> None:
        """Verifica che la directory esista prima di caricare i dati."""
        if not os.path.isdir(path):
            raise ValueError(f"Cartella non trovata: {path}")

    def load_data_config(self):
        """
        Recupera la configurazione nativa del modello pretrained da timm:
        - mean e std usati durante il pretraining su ImageNet
        - input_size attesa dal backbone (es. 224x224 per EfficientNet-B0)
        - crop_pct: percentuale di crop per la valutazione
        - interpolation: metodo di resize (bilinear, bicubic...)

        È fondamentale usare le stesse normalizzazioni del pretraining:
        i feature extractor pretrained sono ottimizzati per queste statistiche.
        Usare statistiche diverse riduce le performance del transfer learning.
        """
        model_for_cfg = timm.create_model(self.config.model_name, pretrained=True)
        self.data_cfg = resolve_data_config({}, model=model_for_cfg)
        return self.data_cfg

    def get_transforms(self):
        """
        Crea le trasformazioni usando timm.create_transform.
        Il livello di augmentation è controllato da config.augmentation.

        Tre livelli disponibili:
        - basic:    flip orizzontale (50%) — minima distorsione, adatto alla baseline
        - moderate: flip + color jitter + RandAugment leggero + random erasing (10%)
        - strong:   flip + vflip + color jitter forte + RandAugment intenso + random erasing (20%)

        IMPORTANTE: le trasformazioni di augmentation vengono applicate SOLO al training set.
        Per validation e test si usa sempre eval_tfms (solo resize e normalizzazione),
        perché il test set deve rappresentare le condizioni reali di deployment.
        """
        if self.data_cfg is None:
            self.load_data_config()

        # Parametri condivisi tra training e valutazione:
        # normalizzazione e dimensioni devono essere identiche al pretraining
        common_kwargs = {
            "input_size": self.data_cfg["input_size"],
            "mean": self.data_cfg["mean"],
            "std": self.data_cfg["std"],
            "interpolation": self.data_cfg["interpolation"],
            "crop_pct": self.data_cfg["crop_pct"],
        }

        if self.config.augmentation == "basic":
            # Solo flip orizzontale: simula che il fiore possa essere fotografato
            # da destra o da sinistra. Nessuna distorsione di colore o geometria.
            train_tfms = create_transform(
                is_training=True,
                hflip=0.5,
                vflip=0.0,
                color_jitter=0.0,
                auto_augment=None,
                re_prob=0.0,
                **common_kwargs
            )
        elif self.config.augmentation == "moderate":
            # RandAugment (rand-m7): policy automatica di augmentation con magnitudo 7/10.
            # re_prob=0.10 → random erasing: cancella una regione casuale dell'immagine (10% delle volte)
            # Simula occlusioni parziali del fiore (foglie, ombra, altri oggetti).
            train_tfms = create_transform(
                is_training=True,
                hflip=0.5,
                vflip=0.0,
                color_jitter=0.2,
                auto_augment="rand-m7-mstd0.5-inc1",
                re_prob=0.10,
                re_mode="pixel",
                re_count=1,
                **common_kwargs
            )
        elif self.config.augmentation == "strong":
            # Augmentation più aggressiva: magnitudo 9/10, random erasing 20%.
            # vflip=0.1: fiori fotografati dall'alto possono apparire capovolti.
            # Aumenta la variabilità del training set → riduce overfitting su dataset piccoli.
            train_tfms = create_transform(
                is_training=True,
                hflip=0.5,
                vflip=0.1,
                color_jitter=0.3,
                auto_augment="rand-m9-mstd0.5-inc1",
                re_prob=0.20,
                re_mode="pixel",
                re_count=1,
                **common_kwargs
            )
        else:
            raise ValueError("augmentation deve essere 'basic', 'moderate' o 'strong'")

        # Trasformazioni di valutazione: solo resize e normalizzazione standard
        eval_tfms = create_transform(
            is_training=False,
            **common_kwargs
        )
        return train_tfms, eval_tfms

    def _is_valid_image_file(self, filename):
        # Filtra file non-immagine come i metadati macOS "._filename"
        # e accetta solo estensioni immagine standard
        return (filename.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp'))
                and not os.path.basename(filename).startswith('._'))

    def build_dataloaders(self):
        """
        Costruisce dataset e dataloader per train, validation e test.

        ImageFolder rileva automaticamente le classi dalle sotto-cartelle:
            dataset_root/train/daisy/     → classe 0
            dataset_root/train/dandelion/ → classe 1
        L'ordine alfabetico determina l'indice numerico della classe.

        is_valid_file: filtra eventuali file nascosti o metadati non-immagine
        presenti nel dataset (comune nei dataset scaricati da macOS/S3).
        """
        train_path = os.path.join(self.config.dataset_root, self.config.train_dir)
        val_path   = os.path.join(self.config.dataset_root, self.config.val_dir)
        test_path  = os.path.join(self.config.dataset_root, self.config.test_dir)

        self._check_dir(train_path)
        self._check_dir(val_path)
        self._check_dir(test_path)

        train_tfms, eval_tfms = self.get_transforms()

        train_dataset = ImageFolder(train_path, transform=train_tfms, is_valid_file=self._is_valid_image_file)
        val_dataset   = ImageFolder(val_path,   transform=eval_tfms,  is_valid_file=self._is_valid_image_file)
        test_dataset  = ImageFolder(test_path,  transform=eval_tfms,  is_valid_file=self._is_valid_image_file)

        # Verifica di consistenza: le classi devono coincidere tra i tre split
        if train_dataset.classes != val_dataset.classes or train_dataset.classes != test_dataset.classes:
            raise ValueError("Le classi di train, validation e test non coincidono.")

        self.class_names = train_dataset.classes

        # shuffle=True solo per il training: mescola i campioni ad ogni epoca
        # per evitare che il modello impari l'ordine dei dati.
        # pin_memory=True: prepara i tensori in memoria pinned per un trasferimento
        # più veloce alla GPU (utile solo se si usa CUDA).
        self.train_loader = DataLoader(
            train_dataset,
            batch_size=self.config.batch_size,
            shuffle=True,
            num_workers=self.config.num_workers,
            pin_memory=torch.cuda.is_available(),
            # drop_last: Mixup richiede batch pari
            # (1275 campioni % 32 = 27 dispari -> AssertionError senza drop_last)
            drop_last=(self.config.mixup_alpha > 0)
        )

        self.val_loader = DataLoader(
            val_dataset,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=self.config.num_workers,
            pin_memory=torch.cuda.is_available()
        )

        self.test_loader = DataLoader(
            test_dataset,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=self.config.num_workers,
            pin_memory=torch.cuda.is_available()
        )

        return train_dataset, val_dataset, test_dataset

    # ----------------------------------------------------------
    # Costruzione del modello
    # ----------------------------------------------------------

    def build_model(self):
        """
        Crea il modello pretrained con timm e adatta la testa finale.

        Transfer learning: invece di allenare una CNN da zero (costoso e lento),
        partiamo da un modello già allenato su ImageNet (1.2M immagini, 1000 classi).
        Il backbone ha già imparato feature visive gerarchiche molto utili:
        - Layer bassi: bordi, texture, colori
        - Layer medi: forme, pattern complessi
        - Layer alti: parti di oggetti (petali, foglie, steli)

        num_classes=2 sostituisce automaticamente la testa finale originale
        (1000 classi per ImageNet) con una nuova testa per 2 classi.

        freeze_backbone=True → congela tutti i pesi del backbone:
        - Solo la nuova testa viene allenata
        - Richiede meno dati e meno tempo di training
        - Utile quando il dataset è piccolo e simile a ImageNet (es. foto di fiori)
        """
        self.model = timm.create_model(
            self.config.model_name,
            pretrained=True,        # pesi pretrained su ImageNet
            num_classes=self.config.num_classes
        )

        if self.config.freeze_backbone:
            # Congela tutti i parametri del backbone
            for param in self.model.parameters():
                param.requires_grad = False

            # Riattiva solo i parametri della testa finale (classifier/fc/head)
            # perché questa è la parte che deve adattarsi al nostro problema
            for name, param in self.model.named_parameters():
                if any(key in name for key in ["classifier", "fc", "head"]):
                    param.requires_grad = True

        self.model = self.model.to(self.device)
        return self.model

    def unfreeze_last_layers(self):
        """
        Sblocca progressivamente gli ultimi layer del backbone.

        Strategia di fine-tuning progressivo:
        Fase 1 (epoche 1-3): backbone congelato → allena solo la testa
        Fase 2 (epoca 4+): sblocca gli ultimi layer del backbone

        Vantaggio: evita che un learning rate alto all'inizio "distrugga" i
        pesi pretrained del backbone. La testa impara prima a produrre
        rappresentazioni stabili, poi il backbone si adatta gradualmente.
        """
        if self.model is None:
            raise ValueError("Il modello non è ancora stato costruito.")

        for name, param in self.model.named_parameters():
            if any(key in name for key in ["blocks", "layer4", "head", "classifier", "fc"]):
                param.requires_grad = True

    # ----------------------------------------------------------
    # Loss, ottimizzatore, scheduler
    # ----------------------------------------------------------

    def build_mixup(self):
        """
        Configura Mixup se mixup_alpha > 0.

        Mixup è una tecnica di augmentation che crea campioni sintetici
        mescolando coppie di immagini e le loro etichette:
            img_mix = λ·img_a + (1-λ)·img_b
            label_mix = λ·label_a + (1-λ)·label_b

        dove λ ~ Beta(alpha, alpha). Con alpha=0.2, λ è quasi sempre vicino a 0 o 1.
        Riduce l'overfitting forzando il modello ad imparare transizioni smooth tra classi.
        Richiede SoftTargetCrossEntropy come loss (perché le etichette non sono più one-hot).
        """
        if self.config.mixup_alpha > 0:
            self.mixup_fn = Mixup(
                mixup_alpha=self.config.mixup_alpha,
                cutmix_alpha=0.0,
                prob=1.0,
                switch_prob=0.0,
                mode="batch",
                label_smoothing=self.config.label_smoothing,
                num_classes=self.config.num_classes
            )
        else:
            self.mixup_fn = None

    def build_criterion(self):
        """
        Sceglie la loss function in base alla configurazione.

        - Mixup attivo        → SoftTargetCrossEntropy (etichette morbide da Mixup)
        - Solo label_smoothing→ LabelSmoothingCrossEntropy (etichette parzialmente morbide)
        - Nessuna tecnica     → CrossEntropyLoss standard

        Label Smoothing (smoothing=0.1):
        Invece di target "hard" [0, 1] usa target "soft" [0.05, 0.95].
        Previene l'overconfidence del modello e migliora la generalizzazione.
        """
        if self.config.mixup_alpha > 0:
            self.train_criterion = SoftTargetCrossEntropy()
        elif self.config.label_smoothing > 0:
            self.train_criterion = LabelSmoothingCrossEntropy(
                smoothing=self.config.label_smoothing
            )
        else:
            self.train_criterion = nn.CrossEntropyLoss()

    def build_optimizer(self):
        """
        Crea l'ottimizzatore AdamW sui soli parametri trainable.

        AdamW vs SGD:
        - AdamW adatta automaticamente il learning rate per ogni parametro
        - Particolarmente efficace per il fine-tuning di modelli pretrained
        - weight_decay in AdamW è applicato correttamente (non come in Adam standard)

        Filtrare solo i parametri con requires_grad=True è essenziale:
        ottimizzare i parametri congelati spreca memoria e tempo.
        """
        trainable_params = [p for p in self.model.parameters() if p.requires_grad]
        self.optimizer = optim.AdamW(
            trainable_params,
            lr=self.config.lr,
            weight_decay=self.config.weight_decay
        )

    def build_scheduler(self):
        """
        Configura lo scheduler del learning rate.

        ReduceLROnPlateau (default):
        - Monitora il F1-score di validation (mode='max')
        - Se non migliora per 2 epoche → moltiplica lr × 0.5
        - Vantaggio: adattivo, riduce lr solo quando serve
        - Uso: adatto per training con early stopping

        CosineAnnealingLR (alternativa):
        - Riduce lr seguendo una curva cosinusoidale da lr_max a lr_min
        - Indipendente dalle metriche di validazione
        - Uso: adatto quando si conosce in anticipo il numero di epoche
        """
        if self.config.use_cosine_scheduler:
            self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=self.config.epochs
            )
        else:
            self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer,
                mode="max",       # monitora una metrica da massimizzare (F1)
                factor=0.5,       # moltiplica lr × 0.5 ad ogni riduzione
                patience=2        # aspetta 2 epoche senza miglioramento prima di ridurre
            )

    # ----------------------------------------------------------
    # Metriche
    # ----------------------------------------------------------

    def compute_metrics(self, y_true, y_pred):
        """
        Calcola accuracy e F1-score macro.

        F1-score macro: media del F1 per ogni classe, senza ponderazione per frequenza.
        È la metrica principale perché valuta il modello in modo bilanciato su entrambe
        le classi, anche se una fosse più frequente dell'altra.
        """
        acc = accuracy_score(y_true, y_pred)
        f1_macro = f1_score(y_true, y_pred, average="macro")
        return acc, f1_macro

    # ----------------------------------------------------------
    # Training / validazione
    # ----------------------------------------------------------

    def train_one_epoch(self):
        """
        Esegue un'epoca di training.

        I 5 passi fondamentali per ogni batch:
        1. Trasferimento immagini e label sul device (GPU)
        2. Eventuale applicazione di Mixup (crea campioni sintetici)
        3. Forward pass con AMP (autocast riduce precisione dove sicuro)
        4. Backpropagation scalata (scaler evita underflow con AMP)
        5. Aggiornamento pesi e reset del gradiente

        model.train() attiva: Dropout (azzera neuroni casuali) e BatchNorm
        (usa statistiche del batch corrente). Questi devono essere attivi
        durante il training ma disattivati durante la valutazione.
        """
        self.model.train()

        running_loss = 0.0
        y_true = []
        y_pred = []

        for images, labels in self.train_loader:
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)

            # Salva le etichette originali prima di applicare Mixup
            # (Mixup produce etichette morbide usate solo per la loss,
            # non per calcolare le metriche di classificazione)
            original_labels = labels.clone()

            if self.mixup_fn is not None:
                images, labels = self.mixup_fn(images, labels)

            # zero_grad() prima di ogni batch: PyTorch accumula i gradienti
            # per default; non azzerarli causerebbe la somma con i gradienti
            # del batch precedente (errore comune per i principianti)
            self.optimizer.zero_grad()

            # autocast: usa float16 per le operazioni supportate (Conv2d, Linear...)
            # riducendo memoria e accelerando il calcolo sulla GPU
            with torch.cuda.amp.autocast(enabled=(self.config.use_amp and self.device.type == "cuda")):
                outputs = self.model(images)
                loss = self.train_criterion(outputs, labels)

            # scaler.scale(loss).backward(): calcola i gradienti scalando la loss
            # per evitare underflow numerici con float16
            self.scaler.scale(loss).backward()
            # scaler.step(): de-scala i gradienti e aggiorna i pesi
            self.scaler.step(self.optimizer)
            # scaler.update(): adatta il fattore di scala per il prossimo step
            self.scaler.update()

            running_loss += loss.item() * images.size(0)
            # argmax: prende l'indice della classe con il logit più alto
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(original_labels.detach().cpu().numpy())
            y_pred.extend(preds.detach().cpu().numpy())

        epoch_loss = running_loss / len(self.train_loader.dataset)
        epoch_acc, epoch_f1 = self.compute_metrics(y_true, y_pred)

        return epoch_loss, epoch_acc, epoch_f1

    @torch.no_grad()
    def evaluate_loader(self, loader):
        """
        Esegue una passata di valutazione su un DataLoader (val o test).

        @torch.no_grad(): disabilita il calcolo del grafo dei gradienti.
        Risparmia ~30-50% di memoria e velocizza l'esecuzione perché
        la backpropagation non è necessaria in fase di valutazione.

        model.eval(): disattiva Dropout (tutti i neuroni attivi) e
        BatchNorm (usa statistiche globali accumulate nel training).
        Queste modalità devono essere disattivate in valutazione per
        ottenere previsioni deterministiche e stabili.
        """
        self.model.eval()

        running_loss = 0.0
        y_true = []
        y_pred = []

        for images, labels in loader:
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(self.config.use_amp and self.device.type == "cuda")):
                outputs = self.model(images)
                # In valutazione si usa sempre eval_criterion (CrossEntropyLoss standard),
                # NON la train_criterion che potrebbe includere Mixup/LabelSmoothing
                loss = self.eval_criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.detach().cpu().numpy())
            y_pred.extend(preds.detach().cpu().numpy())

        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc, epoch_f1 = self.compute_metrics(y_true, y_pred)

        return {
            "loss": epoch_loss,
            "acc": epoch_acc,
            "f1_macro": epoch_f1,
            "y_true": y_true,
            "y_pred": y_pred
        }

    def run(self):
        """
        Esegue l'intero ciclo dell'esperimento:
        1. Preparazione (seed, data config, dataloaders, modello, optimizer...)
        2. Loop per ogni epoca: train_one_epoch → evaluate_loader(val) → scheduler step
        3. Early stopping: salva il miglior modello, termina se non migliora
        4. Ripristino del miglior modello al termine

        METRICA DI SELEZIONE: F1-score macro su validation set.
        Il test set NON viene mai toccato durante questo loop.

        Restituisce un dizionario con il modello migliore, lo storico
        delle metriche per ogni epoca, i loader e le statistiche finali.
        """
        self.set_seed()
        self.load_data_config()
        train_dataset, val_dataset, test_dataset = self.build_dataloaders()
        self.build_model()
        self.build_mixup()
        self.build_criterion()
        self.build_optimizer()
        self.build_scheduler()

        # Storico delle metriche per ogni epoca (usato per i grafici)
        history = {
            "train_loss": [], "train_acc": [], "train_f1": [],
            "val_loss": [],   "val_acc": [],   "val_f1": []
        }

        best_state = None          # state_dict del miglior modello trovato
        best_val_f1 = -1.0        # miglior F1 di validation visto finora
        best_epoch = 0            # epoca in cui è stato trovato il miglior F1
        early_stop_counter = 0    # contatore per l'early stopping

        print(f"Esperimento: {self.config.experiment_name}")
        print(f"Classi: {self.class_names}")
        print(f"Train: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}")

        for epoch in range(1, self.config.epochs + 1):
            start_time = time.time()

            # Fine-tuning progressivo: sblocca il backbone all'epoca specificata
            if self.config.progressive_unfreeze and epoch == self.config.unfreeze_epoch:
                print(f"\n[INFO] Sblocco progressivo attivato all'epoca {epoch}.")
                self.unfreeze_last_layers()
                # Ricostruiamo optimizer e scheduler per includere i nuovi parametri
                self.build_optimizer()
                self.build_scheduler()

            train_loss, train_acc, train_f1 = self.train_one_epoch()
            val_metrics = self.evaluate_loader(self.val_loader)

            # ReduceLROnPlateau vuole il valore della metrica (F1);
            # CosineAnnealingLR non richiede argomenti
            if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                self.scheduler.step(val_metrics["f1_macro"])
            else:
                self.scheduler.step()

            # Aggiornamento dello storico per i grafici
            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["train_f1"].append(train_f1)
            history["val_loss"].append(val_metrics["loss"])
            history["val_acc"].append(val_metrics["acc"])
            history["val_f1"].append(val_metrics["f1_macro"])

            elapsed = time.time() - start_time
            print(f"\nEpoca [{epoch}/{self.config.epochs}] - {elapsed:.1f}s")
            print(f"Train -> Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f} | F1 Macro: {train_f1:.4f}")
            print(f"Val   -> Loss: {val_metrics['loss']:.4f} | Accuracy: {val_metrics['acc']:.4f} | F1 Macro: {val_metrics['f1_macro']:.4f}")

            # Early stopping: salva il checkpoint se il F1 di validation migliora
            if val_metrics["f1_macro"] > best_val_f1:
                best_val_f1 = val_metrics["f1_macro"]
                best_epoch = epoch
                # Salva una copia profonda dello state_dict (solo i pesi numerici)
                best_state = copy.deepcopy(self.model.state_dict())
                early_stop_counter = 0
            else:
                early_stop_counter += 1

            if early_stop_counter >= self.config.patience:
                print(f"\n[INFO] Early stopping attivato. Nessun miglioramento da {self.config.patience} epoche.")
                break

        # Ripristina i pesi del miglior modello trovato durante il training
        if best_state is not None:
            self.model.load_state_dict(best_state)
            torch.save(best_state, self.config.best_model_path)
            print(f"\nMiglior modello salvato in: {self.config.best_model_path}")
            print(f"Miglior epoca: {best_epoch} | Miglior Val F1: {best_val_f1:.4f}")

        return {
            "model": self.model,
            "history": history,
            "criterion": self.eval_criterion,
            "train_loader": self.train_loader,
            "val_loader": self.val_loader,
            "test_loader": self.test_loader,
            "class_names": self.class_names,
            "best_epoch": best_epoch,
            "best_val_f1": best_val_f1
        }


In [ ]:

# ============================================================
# CELLA 6 — Evaluator: valutazione e visualizzazione dei risultati
# ============================================================
# Separare la logica di valutazione dall'ExperimentRunner permette di:
# 1. Valutare qualsiasi modello su qualsiasi loader (val o test) indipendentemente
# 2. Confrontare esperimenti diversi con la stessa interfaccia
# 3. Mantenere il codice di visualizzazione separato da quello di training
# ============================================================

class Evaluator:
    def __init__(self, model, loader, criterion, device, class_names, split_name="validation", use_amp=True):
        self.model = model
        self.loader = loader
        self.criterion = criterion
        self.device = device
        self.class_names = class_names
        self.split_name = split_name
        self.use_amp = use_amp

    def plot_history(self, history):
        """
        Visualizza l'andamento di loss e F1-score macro durante il training.

        Due grafici affiancati:
        - Sinistra: loss di training vs validation (ideale: entrambe decrescenti)
        - Destra: F1 macro di training vs validation (ideale: entrambi crescenti)

        Pattern da cercare:
        - Overfitting: train_loss scende, val_loss risale → modello memorizza i dati
        - Underfitting: entrambe le loss restano alte → modello troppo semplice
        - Buon fit: entrambe le curve convergono a valori bassi
        """
        epochs = range(1, len(history["train_loss"]) + 1)

        plt.figure(figsize=(12, 4))

        plt.subplot(1, 2, 1)
        plt.plot(epochs, history["train_loss"], label="train_loss")
        plt.plot(epochs, history["val_loss"],   label="val_loss")
        plt.xlabel("Epoca")
        plt.ylabel("Loss")
        plt.title("Andamento della loss")
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(epochs, history["train_f1"], label="train_f1_macro")
        plt.plot(epochs, history["val_f1"],   label="val_f1_macro")
        plt.xlabel("Epoca")
        plt.ylabel("F1 Macro")
        plt.title("Andamento del F1 Macro")
        plt.legend()

        plt.tight_layout()
        plt.show()

    @torch.no_grad()
    def evaluate(self):
        """
        Esegue la valutazione completa del modello sul loader specificato.

        @torch.no_grad(): disabilita il calcolo del gradiente → più veloce e
        meno memoria. Non serve la backpropagation in fase di valutazione.

        Restituisce un dizionario con:
        - loss: cross-entropy media su tutti i campioni
        - accuracy: percentuale di predizioni corrette
        - f1_macro: F1-score medio tra le classi (metrica principale)
        - y_true, y_pred: etichette reali e predette (per classification report e matrice)
        """
        self.model.eval()

        running_loss = 0.0
        y_true = []
        y_pred = []

        for images, labels in self.loader:
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(self.use_amp and self.device.type == "cuda")):
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            # argmax lungo dim=1: prende la classe col logit più alto per ogni campione
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

        loss_value = running_loss / len(self.loader.dataset)
        acc_value  = accuracy_score(y_true, y_pred)
        f1_value   = f1_score(y_true, y_pred, average="macro")

        return {
            "loss": loss_value,
            "accuracy": acc_value,
            "f1_macro": f1_value,
            "y_true": y_true,
            "y_pred": y_pred
        }

    def plot_confusion_matrix(self, y_true, y_pred):
        """
        Visualizza la matrice di confusione.

        La matrice di confusione mostra:
        - Diagonale principale: predizioni corrette (TP per ogni classe)
        - Fuori diagonale: errori di classificazione

        Lettura per una classificazione binaria:
              | Pred: daisy | Pred: dandelion
        ------+-------------+----------------
        daisy |    TP_d     |    FN_d        ← falsi negativi daisy
        dand. |    FP_d     |    TP_dan
        """
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=self.class_names)

        fig, ax = plt.subplots(figsize=(6, 6))
        disp.plot(ax=ax, cmap="Blues", colorbar=False)
        plt.title(f"Matrice di confusione - {self.split_name}")
        plt.tight_layout()
        plt.show()

    def full_report(self, history=None, show_cm=True):
        """
        Esegue la valutazione completa e produce tutti i report.

        1. Grafico loss e F1 durante il training (se history è fornito)
        2. Metriche aggregate: loss, accuracy, F1 macro
        3. Classification report per classe: precision, recall, F1, support
        4. Matrice di confusione (se show_cm=True)

        Classification report:
        - Precision: su tutto ciò che il modello predice come classe X,
          quanto spesso ha ragione? (TP / (TP + FP))
        - Recall: su tutti i campioni reali della classe X,
          quanti ne trova il modello? (TP / (TP + FN))
        - F1: media armonica di precision e recall
        - Support: numero di campioni reali per classe
        """
        if history is not None:
            self.plot_history(history)

        metrics = self.evaluate()

        print("\n" + "=" * 70)
        print(f"RISULTATI SU {self.split_name.upper()}")
        print("=" * 70)
        print(f"Loss:      {metrics['loss']:.4f}")
        print(f"Accuracy:  {metrics['accuracy']:.4f}")
        print(f"F1 Macro:  {metrics['f1_macro']:.4f}")

        print("\nClassification Report:")
        print(classification_report(
            metrics["y_true"],
            metrics["y_pred"],
            target_names=self.class_names,
            digits=4
        ))

        if show_cm:
            self.plot_confusion_matrix(metrics["y_true"], metrics["y_pred"])

        return metrics


In [ ]:

# ============================================================
# CELLA 7 — Configurazione degli esperimenti
# ============================================================
# Ogni esperimento modifica UN solo aspetto rispetto alla baseline,
# seguendo il principio del "controllo delle variabili":
# se cambiassimo più parametri insieme, non sapremmo quale ha fatto la differenza.
#
# Tutti gli esperimenti usano:
# - efficientnet_b0 come backbone
# - AdamW come ottimizzatore
# - ReduceLROnPlateau come scheduler
# - early stopping con patience=4
# ============================================================

# ---------------------------------------------------------------
# Esperimento 1: BASELINE
# Scopo: stabilire un punto di riferimento minimo.
# Backbone congelato → allena solo la testa lineare finale.
# Solo flip orizzontale come augmentation.
# Nessuna regolarizzazione aggiuntiva.
# lr=1e-3: più alto perché la testa è inizializzata casualmente.
# ---------------------------------------------------------------
baseline_cfg = Config(
    experiment_name="baseline_basic",
    model_name="efficientnet_b0",
    freeze_backbone=True,
    progressive_unfreeze=False,
    augmentation="basic",
    label_smoothing=0.0,
    mixup_alpha=0.0,
    lr=1e-3,
    best_model_path="best_baseline_basic.pth"
)

# ---------------------------------------------------------------
# Esperimento 2: FINE-TUNING PROGRESSIVO
# Scopo: verificare se lo sblocco graduale del backbone migliora le performance.
# Strategia a 2 fasi:
#   Fase 1 (ep. 1-3): backbone congelato, allena solo la testa → stabilizza le rappresentazioni
#   Fase 2 (ep. 4+):  sblocca gli ultimi layer del backbone → adatta al dominio fiori
# augmentation="moderate": aggiunge color jitter + RandAugment + random erasing
# ---------------------------------------------------------------
progressive_cfg = Config(
    experiment_name="progressive_moderate",
    model_name="efficientnet_b0",
    freeze_backbone=True,
    progressive_unfreeze=True,
    unfreeze_epoch=4,
    augmentation="moderate",
    label_smoothing=0.0,
    mixup_alpha=0.0,
    lr=1e-3,
    best_model_path="best_progressive_moderate.pth"
)

# ---------------------------------------------------------------
# Esperimento 3: FINE-TUNING COMPLETO
# Scopo: tutto il backbone è allenabile fin dall'inizio.
# lr=5e-4: più basso per evitare di "distruggere" i pesi pretrained del backbone.
# label_smoothing=0.1: target soft [0.05, 0.95] invece di hard [0, 1]
#   → il modello è meno sicuro di sé → generalizza meglio.
# Aumentare la capacità del modello (tutto allenabile) richiede più regolarizzazione.
# ---------------------------------------------------------------
full_ft_cfg = Config(
    experiment_name="full_ft_moderate",
    model_name="efficientnet_b0",
    freeze_backbone=False,
    progressive_unfreeze=False,
    augmentation="moderate",
    label_smoothing=0.1,
    mixup_alpha=0.0,
    lr=5e-4,
    best_model_path="best_full_ft_moderate.pth"
)

# ---------------------------------------------------------------
# Esperimento 4: AUGMENTATION FORTE + LABEL SMOOTHING
# Scopo: massimizzare la robustezza del modello con regolarizzazione più aggressiva.
# augmentation="strong": aggiunge vflip (10%) + color jitter forte + RandAugment magnitudo 9.
# Utile con dataset piccoli: crea più variabilità artificialmente.
# Combinazione con label_smoothing=0.1 per prevenire overconfidence.
# ---------------------------------------------------------------
strong_aug_cfg = Config(
    experiment_name="strong_aug_label_smoothing",
    model_name="efficientnet_b0",
    freeze_backbone=False,
    progressive_unfreeze=False,
    augmentation="strong",
    label_smoothing=0.1,
    mixup_alpha=0.0,
    lr=5e-4,
    best_model_path="best_strong_aug_label_smoothing.pth"
)

# ---------------------------------------------------------------
# Esperimento 5: AUGMENTATION FORTE + MIXUP
# Scopo: aggiungere Mixup al setup migliore per ridurre ulteriormente l'overfitting.
# Mixup (alpha=0.2): crea campioni sintetici mescolando coppie di immagini.
#   img_mix = λ·img_a + (1-λ)·img_b, dove λ ~ Beta(0.2, 0.2)
# Richiede SoftTargetCrossEntropy come loss (etichette non sono più one-hot).
# ---------------------------------------------------------------
mixup_cfg = Config(
    experiment_name="strong_aug_mixup",
    model_name="efficientnet_b0",
    freeze_backbone=False,
    progressive_unfreeze=False,
    augmentation="strong",
    label_smoothing=0.1,
    mixup_alpha=0.2,
    lr=5e-4,
    best_model_path="best_strong_aug_mixup.pth"
)

# Lista di tutti gli esperimenti definiti
experiments = [
    baseline_cfg,
    progressive_cfg,
    full_ft_cfg,
    strong_aug_cfg,
    mixup_cfg
]

print(f"Definiti {len(experiments)} esperimenti:")
for cfg in experiments:
    print(f"  - {cfg.experiment_name}")


In [ ]:

# ============================================================
# TEST 1 — Baseline (backbone congelato, augmentation basic)
# ============================================================
# Questo test stabilisce il punto di partenza del progetto.
# Il backbone di EfficientNet-B0 è pretrained su ImageNet: i pesi
# non vengono modificati. Si allena solo la testa finale (1280→2 neuroni).
#
# Cosa ci aspettiamo:
# - Training veloce (pochi parametri allenabili)
# - Buone performance grazie al transfer learning
# - Possibile margine di miglioramento rispetto alle configurazioni con fine-tuning
#
# NOTA: usiamo val_loader (NON test_loader) per il confronto tra esperimenti.
# Il test set va usato SOLO per la valutazione finale del modello scelto.
# ============================================================

import copy
config = copy.deepcopy(baseline_cfg)

# Aggiornamento del percorso dataset dopo lo spostamento in datasets/
config.dataset_root = os.path.join("datasets", config.dataset_root)

runner = ExperimentRunner(config, device=device)
result = runner.run()

# Valutazione su VALIDATION set (non test set!) per confronto equo tra esperimenti
evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],   # CORRETTO: val_loader per il confronto tra esperimenti
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="valid",
    use_amp=config.use_amp
)

baseline_metrics = evaluator.full_report(history=result["history"])
baseline_metrics


## Esperimenti

Ho testato cinque configurazioni progressive, partendo da un baseline semplice e aggiungendo incrementalmente tecniche di fine-tuning e regolarizzazione. La logica è stata volutamente incrementale: ogni esperimento modifica una cosa sola rispetto al precedente, così da isolare l'effetto di ogni scelta tecnica.

La metrica di confronto è il **F1-score macro** sul validation set — più bilanciata dell'accuracy perché bilancia le prestazioni su entrambe le classi (163 daisy vs 201 dandelion nel validation set).

---

### Esperimento 1 — `baseline_basic`

Backbone congelato, solo la testa classificatrice allenata, augmentation leggera. Il punto di partenza da cui misurare tutto il resto.

#### Risultati (validation set)

| Metrica | Valore |
|---------|--------|
| Miglior epoca | 5 (early stopping ep. 9) |
| Val Loss | 0.5291 |
| Val Accuracy | 0.9478 |
| **Val F1 Macro** | **0.9473** |

| Classe | Precision | Recall | F1 |
|--------|-----------|--------|----|
| daisy | 0.9337 | 0.9509 | 0.9422 |
| dandelion | 0.9596 | 0.9453 | 0.9524 |

Matrice di confusione: 155/163 daisy corrette, 190/201 dandelion corrette.

#### Analisi

Il risultato è già alto per un backbone completamente congelato: F1=0.9473 alla quinta epoca. Conferma che EfficientNet-B0 preaddestrato su ImageNet ha già sviluppato feature discriminative per texture e forma — non sorprende che riconosca margherite da tarassachi con un semplice classificatore lineare in cima.

L'early stopping si attiva all'epoca 9 perché il modello non migliora da 4 epoche consecutive. Con backbone congelato la convergenza è rapida ma il potenziale è limitato: la testa lineare esaurisce il suo margine abbastanza presto, e la val loss a 0.5291 segnala che c'è ancora capacità non sfruttata nel backbone — ma per attingervi serve il fine-tuning.

In [ ]:

# ============================================================
# TEST 2 — Fine-tuning progressivo (backbone → sblocco a ep. 4, augmentation moderate)
# ============================================================
# Strategia a 2 fasi:
#   Fase 1 (ep. 1-3): allena solo la testa, backbone congelato
#   Fase 2 (ep. 4+):  sblocca gli ultimi layer → backbone si adatta al dominio fiori
#
# Motivazione dello sblocco graduale:
# Se sbloccassimo subito il backbone con lr=1e-3, i gradienti "distruggerebbero"
# i pesi pretrained di ImageNet prima che la testa sia abbastanza stabile.
# Con il warm-up iniziale, la testa apprende una buona rappresentazione,
# poi il backbone si adatta delicatamente.
#
# progressive_unfreeze=True → ExperimentRunner chiama unfreeze_last_layers()
# all'epoca config.unfreeze_epoch e ricostruisce optimizer e scheduler.
# ============================================================

config = copy.deepcopy(progressive_cfg)
config.dataset_root = os.path.join("datasets", config.dataset_root)

runner = ExperimentRunner(config, device=device)
result = runner.run()

# Valutazione sul validation set per confronto con gli altri esperimenti
evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="valid",
    use_amp=config.use_amp
)

progressive_metrics = evaluator.full_report(history=result["history"])
progressive_metrics



### Esperimento 2 — `progressive_moderate`

Fine-tuning progressivo: prime epoche con backbone congelato, poi sblocco graduale degli ultimi layer. L'idea è evitare di sovrascrivere troppo bruscamente i pesi pretrained — il backbone parte da rappresentazioni già buone e le adatta in modo graduale al dominio dei fiori.

#### Risultati (validation set)

| Metrica | Valore |
|---------|--------|
| Miglior epoca | 12 (su 15 totali) |
| Val Loss | 0.0610 |
| Val Accuracy | 0.9753 |
| **Val F1 Macro** | **0.9751** |

| Classe | Precision | Recall | F1 |
|--------|-----------|--------|----|
| daisy | 0.9529 | 0.9939 | 0.9730 |
| dandelion | 0.9948 | 0.9602 | 0.9772 |

Matrice di confusione: 162/163 daisy corrette (1 solo errore), 193/201 dandelion corrette.

#### Analisi

Il salto rispetto al baseline è netto: F1 da 0.9473 a 0.9751 (+0.0278). La val loss scende da 0.5291 a 0.0610 — adattare i pesi del backbone al dominio dei fiori produce embedding molto più discriminativi.

Nel grafico si vede chiaramente il momento dello sblocco (intorno all'epoca 4): la training loss cala più velocemente e il val F1 riparte verso l'alto. Questo è il comportamento cercato con l'unfreeze progressivo — senza quel warm-up iniziale il backbone rischierebbe di perdere troppo del pretraining nelle prime iterazioni ad alto learning rate.

Il dato più significativo: recall di daisy a 0.9939, solo 1 errore su 163 campioni.

In [ ]:

# ============================================================
# TEST 3 — Fine-tuning completo (tutto il backbone allenabile, augmentation moderate)
# ============================================================
# Differenza chiave rispetto all'esperimento 2:
# - freeze_backbone=False: tutti i layer del backbone sono allenabili fin dall'inizio
# - lr=5e-4 (dimezzato): learning rate più basso per non destabilizzare il backbone
# - label_smoothing=0.1: aggiunge regolarizzazione per compensare la maggiore capacità
#
# Quando conviene il fine-tuning completo?
# - Dataset abbastanza grande (riduce il rischio di overfitting del backbone)
# - Task molto diverso da ImageNet (il backbone deve adattarsi significativamente)
# - Quando il backbone parzialmente sbloccato non è sufficiente
#
# Confrontare con exp 2: se progressive_unfreeze ottiene F1 simile,
# vuol dire che l'approccio progressivo è preferibile (più stabile, meno rischio).
# ============================================================

config = copy.deepcopy(full_ft_cfg)
config.dataset_root = os.path.join("datasets", config.dataset_root)

runner = ExperimentRunner(config, device=device)
result = runner.run()

# Valutazione sul validation set
evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="valid",
    use_amp=config.use_amp
)

full_ft_metrics = evaluator.full_report(history=result["history"])
full_ft_metrics



### Esperimento 3 — `full_ft_moderate`

Fine-tuning completo dall'inizio: tutti i pesi del backbone allenabili fin dalla prima epoca, augmentation moderata e label smoothing. Il contrario diretto dell'approccio progressivo.

#### Risultati (validation set)

| Metrica | Valore |
|---------|--------|
| Miglior epoca | 15 (ha usato tutte le epoche disponibili) |
| Val Loss | 0.1421 |
| Val Accuracy | 0.9670 |
| **Val F1 Macro** | **0.9669** |

| Classe | Precision | Recall | F1 |
|--------|-----------|--------|----|
| daisy | 0.9314 | 1.0000 | 0.9645 |
| dandelion | 1.0000 | 0.9403 | 0.9692 |

Matrice di confusione: 163/163 daisy corrette (zero errori), 189/201 dandelion corrette.

#### Analisi

Il risultato è controintuitivo: sbloccare tutto il backbone subito peggiora rispetto all'approccio progressivo (F1 0.9669 vs 0.9751). La spiegazione più probabile è che con backbone libero fin dall'inizio il classificatore finale non è ancora stabile quando il backbone riceve i primi gradienti — e il learning rate modifica i pesi pretrained in modo troppo aggressivo, perdendo parte del pretraining accumulato.

Un dato curioso: recall di daisy = 1.0000, zero falsi negativi su 163 campioni. Ma dandelion ha 12 errori (recall 0.9403). Il modello ha polarizzato l'attenzione su daisy a scapito di dandelion.

Il fatto che il miglior epoch sia alla 15a su 15 disponibili indica che il modello non ha finito di convergere — probabilmente con 20+ epoche avrebbe superato il progressivo. Ma a parità di budget di epoche, il fine-tuning progressivo è più efficiente.

In [ ]:

# ============================================================
# TEST 4 — Augmentation forte + label smoothing (fine-tuning completo)
# ============================================================
# Configurazione più aggressiva: combina fine-tuning completo con
# augmentation intensa e label smoothing.
#
# augmentation="strong" aggiunge rispetto a "moderate":
# - vflip (10%): fiori fotografati dall'alto appaiono capovolti
# - color jitter più forte (0.3 vs 0.2): simula illuminazioni diverse
# - RandAugment più intenso (m=9 vs m=7): trasformazioni più aggressive
# - Random Erasing più frequente (20% vs 10%): simula occlusioni maggiori
#
# label_smoothing=0.1:
# Trasforma il target da [0, 1] a [0.05, 0.95] (con 2 classi).
# Il modello non può mai essere "troppo sicuro" → generalizza meglio.
# Formula: target_smooth = target × (1 - α) + α / num_classes
# Con α=0.1: target smooth per classe corretta = 0.9 + 0.1/2 = 0.95
#
# Risultato atteso: convergenza più rapida (es. ep. 7) grazie all'augmentation intensa.
# ============================================================

config = copy.deepcopy(strong_aug_cfg)
config.dataset_root = os.path.join("datasets", config.dataset_root)

runner = ExperimentRunner(config, device=device)
result = runner.run()

# Valutazione sul validation set
evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="valid",
    use_amp=config.use_amp
)

strong_aug_metrics = evaluator.full_report(history=result["history"])
strong_aug_metrics



### Esperimento 4 — `strong_aug_label_smoothing`

Fine-tuning completo con augmentation più aggressiva (RandAugment, random erasing, color jitter) e label smoothing (ε=0.1). Rispetto all'esp. 3, il cambiamento principale è nell'intensità delle trasformazioni di input — il backbone è libero in entrambi i casi.

#### Risultati (validation set)

| Metrica | Valore |
|---------|--------|
| Miglior epoca | 15 (ha usato tutte le epoche disponibili) |
| Val Loss | 0.1224 |
| Val Accuracy | 0.9753 |
| **Val F1 Macro** | **0.9751** |

| Classe | Precision | Recall | F1 |
|--------|-----------|--------|----|
| daisy | 0.9583 | 0.9877 | 0.9728 |
| dandelion | 0.9898 | 0.9652 | 0.9773 |

Matrice di confusione: 161/163 daisy corrette, 194/201 dandelion corrette.

#### Analisi

F1=0.9751, identico al fine-tuning progressivo (esp. 2) ma con dinamiche di convergenza molto diverse. La training loss resta alta per tutto il training — effetto diretto dell'augmentation forte: il modello lavora su immagini distorte e non può memorizzarle. Eppure la val loss scende comunque (0.1224), segno che impara feature robuste nonostante il training difficile.

Il confronto con l'esp. 3 è interessante: gli errori su dandelion calano da 12 a 7. L'augmentation più intensa ha migliorato la generalizzazione su quella classe. Però il modello non ha ancora raggiunto il plateau alla 15a epoca — potrebbe crescere ulteriormente, ma a parità di epoche non batte il fine-tuning progressivo.


### Esperimento 5 — `strong_aug_mixup`

Questo esperimento estende la configurazione dell'esperimento 4 aggiungendo la tecnica di **Mixup** come ulteriore forma di regolarizzazione.

**Cos'è Mixup?**  
Mixup crea campioni di training sintetici mescolando coppie di immagini e le loro etichette:

```
img_mix   = λ·img_a + (1-λ)·img_b
label_mix = λ·label_a + (1-λ)·label_b
```

dove `λ ~ Beta(alpha, alpha)`. Con `alpha=0.2`, λ è quasi sempre vicino a 0 o 1, producendo blend leggeri tra le immagini.

**Perché Mixup riduce l'overfitting?**  
Forza il modello ad imparare transizioni "smooth" tra classi, invece di confini netti e rigidi. Questo lo rende più robusto a piccole perturbazioni dell'input.

**Effetto sulle etichette:**  
Le etichette non sono più one-hot `[0, 1]` ma soft `[0.3, 0.7]` → non si può usare la `CrossEntropyLoss` standard. Si usa invece `SoftTargetCrossEntropy` (da timm), che accetta etichette continue.

Scopo:
- verificare se Mixup porta un ulteriore miglioramento rispetto all'augmentation forte e al label smoothing
- confrontare l'effetto di Mixup vs label smoothing da soli vs la loro combinazione


In [ ]:

# ============================================================
# TEST 5 — Augmentation forte + Mixup (alpha=0.2)
# ============================================================
# Aggiunge Mixup alla configurazione dell'esperimento 4 (strong_aug).
#
# mixup_alpha=0.2: λ ~ Beta(0.2, 0.2)
#   - λ vicino a 1.0 (80% dei casi): blend quasi puro di img_a
#   - λ vicino a 0.0 (80% dei casi): blend quasi puro di img_b
#   - λ ≈ 0.5 (raro): blend 50/50 delle due immagini
#
# Nota tecnica: Mixup usa SoftTargetCrossEntropy (build_criterion lo gestisce
# automaticamente quando mixup_alpha > 0). Le metriche di training potrebbero
# sembrare peggiori del solito perché il modello lavora con immagini "corrotte",
# ma la valutazione su validation usa immagini reali non modificate.
#
# Confronto atteso con exp 4:
# - Se F1 simile → Mixup non aggiunge valore (già abbastanza regolarizzato)
# - Se F1 migliore → Mixup porta ulteriore beneficio
# ============================================================

config = copy.deepcopy(mixup_cfg)
config.dataset_root = os.path.join("datasets", config.dataset_root)

runner = ExperimentRunner(config, device=device)
result = runner.run()

# Valutazione sul validation set
evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="valid",
    use_amp=config.use_amp
)

mixup_metrics = evaluator.full_report(history=result["history"])
mixup_metrics

# Salva il risultato del mixup per la valutazione finale sul test set
mixup_result = result


### Analisi Esperimento 5 — `strong_aug_mixup`

| Metrica | Valore (validation) |
|---------|---------------------|
| Loss | 0.1321 |
| Accuracy | 98.08% |
| **F1 Macro** | **0.9806** |

| Classe | Precision | Recall | F1-Score | Support |
|--------|-----------|--------|----------|---------|
| daisy | 0.9643 | 0.9939 | 0.9789 | 163 |
| dandelion | 0.9949 | 0.9701 | 0.9824 | 201 |
| **Macro avg** | **0.9796** | **0.9820** | **0.9806** | 364 |

Matrice di confusione: daisy 162/163 (1 errore), dandelion 195/201 (6 errori).

---

F1 Macro = **0.9806**, il migliore tra tutti gli esperimenti.

Il Mixup aggiunge all'augmentation forte un ulteriore livello di regolarizzazione: invece di trasformare le singole immagini, crea campioni sintetici interpolando linearmente coppie di immagini e le loro etichette (`x_mix = λ·x_i + (1−λ)·x_j`, con λ ~ Beta(0.2, 0.2)). Le etichette risultanti sono morbide — tipo `[0.85, 0.15]` invece di `[1, 0]` — e impediscono al modello di diventare troppo sicuro dei boundary di classificazione, producendo confini di decisione più regolari.

Caratteristica della curva di loss: la training loss resta alta (~0.39 all'ultima epoca) perché i campioni ibridi sono difficili da classificare con etichette morbide. La val loss è 0.1321 e il val F1 è 0.9806. Il gap non segnala overfitting — è normale e atteso con Mixup.

**Nota tecnica:** per funzionare correttamente è stato necessario aggiungere `drop_last=True` nel DataLoader di training. Con 1275 campioni e batch_size=32, l'ultimo batch ha 27 campioni (dispari) e Mixup richiede batch pari per interpolare le coppie. Senza il fix, `AssertionError` a ogni epoca.

### Strategia di selezione del modello
Per tutti gli esperimenti, il modello migliore è stato selezionato in base alle prestazioni sul **validation set**, utilizzando come criterio principale il **F1-score macro**.  
Il **test set** è stato utilizzato solo nella fase finale, così da ottenere una valutazione imparziale delle capacità di generalizzazione del modello scelto.

In [ ]:

# ============================================================
# CONFRONTO ESPERIMENTI — usa i risultati gia' calcolati
# ============================================================
# I 5 esperimenti sono gia' stati eseguiti nelle celle precedenti.
# Le variabili *_metrics contengono i risultati sul validation set.
# NON e' necessario rieseguire il training: usiamo i dati gia' in memoria.
#
# REGOLA: solo il validation set viene usato per confrontare gli esperimenti.
# Il test set verra' usato UNA SOLA VOLTA alla fine, sul modello scelto.
# ============================================================

import pandas as pd

# Raccoglie le metriche di validazione gia' calcolate da ciascun esperimento
# Le variabili *_metrics provengono dalle celle precedenti (Test 1-5)
experiment_results = [
    {
        "Esperimento":      baseline_cfg.experiment_name,
        "Augmentation":     baseline_cfg.augmentation,
        "Fine-tuning":      "solo testa" if baseline_cfg.freeze_backbone else "progressivo" if baseline_cfg.progressive_unfreeze else "completo",
        "Label smoothing":  getattr(baseline_cfg, "label_smoothing", 0.0),
        "Mixup alpha":      getattr(baseline_cfg, "mixup_alpha", 0.0),
        "Val Loss":         round(baseline_metrics["loss"], 4),
        "Val Accuracy":     round(baseline_metrics["accuracy"], 4),
        "Val F1 Macro":     round(baseline_metrics["f1_macro"], 4),
    },
    {
        "Esperimento":      progressive_cfg.experiment_name,
        "Augmentation":     progressive_cfg.augmentation,
        "Fine-tuning":      "solo testa" if progressive_cfg.freeze_backbone else "progressivo" if progressive_cfg.progressive_unfreeze else "completo",
        "Label smoothing":  getattr(progressive_cfg, "label_smoothing", 0.0),
        "Mixup alpha":      getattr(progressive_cfg, "mixup_alpha", 0.0),
        "Val Loss":         round(progressive_metrics["loss"], 4),
        "Val Accuracy":     round(progressive_metrics["accuracy"], 4),
        "Val F1 Macro":     round(progressive_metrics["f1_macro"], 4),
    },
    {
        "Esperimento":      full_ft_cfg.experiment_name,
        "Augmentation":     full_ft_cfg.augmentation,
        "Fine-tuning":      "solo testa" if full_ft_cfg.freeze_backbone else "progressivo" if full_ft_cfg.progressive_unfreeze else "completo",
        "Label smoothing":  getattr(full_ft_cfg, "label_smoothing", 0.0),
        "Mixup alpha":      getattr(full_ft_cfg, "mixup_alpha", 0.0),
        "Val Loss":         round(full_ft_metrics["loss"], 4),
        "Val Accuracy":     round(full_ft_metrics["accuracy"], 4),
        "Val F1 Macro":     round(full_ft_metrics["f1_macro"], 4),
    },
    {
        "Esperimento":      strong_aug_cfg.experiment_name,
        "Augmentation":     strong_aug_cfg.augmentation,
        "Fine-tuning":      "solo testa" if strong_aug_cfg.freeze_backbone else "progressivo" if strong_aug_cfg.progressive_unfreeze else "completo",
        "Label smoothing":  getattr(strong_aug_cfg, "label_smoothing", 0.0),
        "Mixup alpha":      getattr(strong_aug_cfg, "mixup_alpha", 0.0),
        "Val Loss":         round(strong_aug_metrics["loss"], 4),
        "Val Accuracy":     round(strong_aug_metrics["accuracy"], 4),
        "Val F1 Macro":     round(strong_aug_metrics["f1_macro"], 4),
    },
    {
        "Esperimento":      mixup_cfg.experiment_name,
        "Augmentation":     mixup_cfg.augmentation,
        "Fine-tuning":      "solo testa" if mixup_cfg.freeze_backbone else "progressivo" if mixup_cfg.progressive_unfreeze else "completo",
        "Label smoothing":  getattr(mixup_cfg, "label_smoothing", 0.0),
        "Mixup alpha":      getattr(mixup_cfg, "mixup_alpha", 0.0),
        "Val Loss":         round(mixup_metrics["loss"], 4),
        "Val Accuracy":     round(mixup_metrics["accuracy"], 4),
        "Val F1 Macro":     round(mixup_metrics["f1_macro"], 4),
    },
]

summary_df = (
    pd.DataFrame(experiment_results)
    .sort_values("Val F1 Macro", ascending=False)
    .reset_index(drop=True)
)

print("\nRIEPILOGO CONFRONTO ESPERIMENTI (ordinato per Val F1 Macro decrescente):")
print(summary_df.to_string(index=False))

# Identifica il miglior esperimento per la selezione finale del modello
best_row = summary_df.iloc[0]
print(f"\nMIGLIOR ESPERIMENTO: {best_row['Esperimento']} — Val F1 Macro = {best_row['Val F1 Macro']:.4f}")

summary_df


## Risultati degli esperimenti

| Esperimento | Augmentation | Fine-tuning | Regolarizzazione | Val F1 Macro |
|------------|--------------|-------------|-----------------|-------------|
| `baseline_basic` | base | solo testa | — | 0.9473 |
| `progressive_moderate` | moderata | progressivo | label smoothing 0.1 | 0.9751 |
| `full_ft_moderate` | moderata | completo | label smoothing 0.1 | 0.9669 |
| `strong_aug_label_smoothing` | forte | completo | label smoothing 0.1 | 0.9751 |
| **`strong_aug_mixup`** | forte | completo | **Mixup α=0.2** | **0.9806** |

Il Mixup è la tecnica che ha fatto la differenza: F1=0.9806 contro 0.9751 degli esperimenti 2 e 4. Un gap piccolo in valore assoluto (+0.0055), ma consistente.

Una cosa che emerge chiaramente dal confronto è che il fine-tuning progressivo (esp. 2) batte quello completo immediato (esp. 3), nonostante quest'ultimo sembri tecnicamente più potente. Il motivo è che sbloccare il backbone tutto subito non dà al classificatore finale il tempo di stabilizzarsi — il backbone riceve gradienti aggressivi fin dalla prima epoca e distorce i pesi pretrained. L'approccio progressivo, invece, parte da feature già buone e le raffina gradualmente, e si vede: F1 0.9751 vs 0.9669.

Il baseline a 0.9473 è già un risultato rispettabile — EfficientNet-B0 su ImageNet fornisce feature che funzionano bene anche per i fiori senza alcun fine-tuning. Tutto il lavoro successivo serve a guadagnare quegli ulteriori ~3 punti di F1.

Il modello selezionato per la valutazione finale è **`strong_aug_mixup`** (Val F1=0.9806). Il test set viene usato solo nella sezione seguente, per una stima imparziale delle performance reali.

In [ ]:

# ============================================================
# VALUTAZIONE FINALE SUL TEST SET — Esperimento: strong_aug_mixup
# ============================================================
# Il test set viene usato QUI per la prima e unica volta.
#
# Modello scelto: strong_aug_mixup (Val F1 Macro = 0.9806, il migliore)
#
# Non riaddestro il modello: uso i pesi gia' salvati durante il Test 5.
# mixup_result["model"] contiene gia' i pesi del miglior checkpoint
# (caricati automaticamente da ExperimentRunner alla fine del training).
#
# REGOLA anti-data-leakage:
# Il test set non e' mai stato visto durante il training ne' durante
# la selezione del modello (quella si e' basata sul validation set).
# Usarlo ora garantisce una stima imparziale delle performance reali.
# ============================================================

# Valutazione FINALE sul test set con il modello gia' addestrato
test_evaluator = Evaluator(
    model=mixup_result["model"],           # pesi del miglior checkpoint gia' caricati
    loader=mixup_result["test_loader"],    # test set: usato per la prima volta
    criterion=mixup_result["criterion"],
    device=device,
    class_names=mixup_result["class_names"],
    split_name="test",
    use_amp=mixup_cfg.use_amp
)

test_metrics_final = test_evaluator.full_report(history=mixup_result["history"])
test_metrics_final


## Valutazione finale sul test set

Il modello **`strong_aug_mixup`** è quello con il miglior F1 Macro sul validation set (0.9806) ed è valutato qui per la prima e unica volta sul test set — nessun parametro è stato modificato dopo aver visto i risultati di validation.

I risultati sono generati dalla cella di codice precedente.

Se il gap tra val F1 (0.9806) e test F1 è contenuto, il modello ha generalizzato bene e la selezione sul validation non ha introdotto selection bias significativo. Un gap più ampio indicherebbe che la scelta del modello si è "adattata" troppo al validation set specifico — ma con un dataset bilanciato e un validation set dello stesso ~20% del totale, questo rischio è limitato.

## Conclusione

Partendo da un baseline con backbone congelato (F1=0.9473), cinque configurazioni progressive hanno portato a un classificatore con F1 Macro=0.9806 sul validation set — circa 3 punti percentuali di miglioramento.

La tecnica più efficace è il **Mixup**: etichette morbide e interpolazione dei campioni producono boundary di decisione più regolari, con un vantaggio misurabile rispetto all'augmentation forte con label smoothing (+0.0055 di F1 Macro).

Un risultato da sottolineare: il fine-tuning progressivo (sblocco graduale del backbone) batte il fine-tuning completo immediato, nonostante quest'ultimo sembri tecnicamente più potente. In pratica, dare al classificatore finale qualche epoca per stabilizzarsi prima di liberare il backbone protegge i pesi pretrained e porta a convergenza migliore. È una lezione abbastanza generale del transfer learning su dataset di dimensioni contenute.

| Esperimento | Val F1 |
|-------------|--------|
| `baseline_basic` | 0.9473 |
| `full_ft_moderate` | 0.9669 |
| `progressive_moderate` | 0.9751 |
| `strong_aug_label_smoothing` | 0.9751 |
| **`strong_aug_mixup`** | **0.9806** |

Il sistema risponde al requisito di GreenTech Solutions Ltd. per il monitoraggio automatico delle colture: le performance reali sul test set sono disponibili nella sezione precedente.